In [1]:
!git clone https://github.com/bremsstrahlung-57/practicum-project

fatal: destination path 'practicum-project' already exists and is not an empty directory.


In [2]:
!pip install torch_pruning -q

ERROR: Operation cancelled by user


In [3]:
from google.colab import userdata
WANDB_API_KEY = userdata.get('WANDB_API_KEY')
import wandb
wandb.login(WANDB_API_KEY)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sagar-sharma-03015611924 (practicum-project) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [15]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch_pruning as tp
import numpy as np
from torchvision.models import resnet18
import os

In [5]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                         download=True, transform=transform_train)
testset  = torchvision.datasets.CIFAR10(root='./data', train=False,
                                         download=True, transform=transform_test)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=128,
                                           shuffle=True, num_workers=2)
testloader  = torch.utils.data.DataLoader(testset, batch_size=256,
                                           shuffle=False, num_workers=2)

100%|██████████| 170M/170M [15:01<00:00, 189kB/s]


In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Rebuild baseline arch
def get_cifar_resnet18():
    model = resnet18(weights=None)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(512, 10)
    return model

base_model = get_cifar_resnet18().to(device)

# Re-apply pruning to get the right architecture shape
example_input = torch.randn(1, 3, 32, 32).to(device)
pruner = tp.pruner.MagnitudePruner(
    base_model,
    example_input,
    importance=tp.importance.MagnitudeImportance(p=1),
    ch_sparsity=0.5,
    ignored_layers=[base_model.fc],
)
pruner.step()

# Now load your saved weights into this shell
checkpoint = torch.load(
    '/content/practicum-project/models/structured_pruning/pruned/structured_pruned_50pct_fp32.pth',
    map_location=device
)
base_model.load_state_dict(checkpoint['model_state_dict'])
model = base_model.to(device)

In [17]:
print(f"{(sum(p.numel() for p in model.parameters()) / 1e6):.2f}M")
torch.save(model.state_dict(), '/tmp/_tmp_model.pth')
size = os.path.getsize('/tmp/_tmp_model.pth') / 1024**2
os.remove('/tmp/_tmp_model.pth')
print(f"{(size):.2f}MB")

2.80M
10.73MB


In [18]:
LR        = 1e-4
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3,
                            momentum=0.9, weight_decay=5e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=40, eta_min=1e-6)
EPOCHS = 40

In [19]:
# Add CutMix to your train loop
def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size(0)
    rand_index = torch.randperm(batch_size).to(x.device)

    y_a, y_b = y, y[rand_index]

    cx = np.random.randint(32)
    cy = np.random.randint(32)
    cut_w = int(32 * np.sqrt(1 - lam))
    cut_h = int(32 * np.sqrt(1 - lam))

    x1 = max(cx - cut_w // 2, 0)
    x2 = min(cx + cut_w // 2, 32)
    y1 = max(cy - cut_h // 2, 0)
    y2 = min(cy + cut_h // 2, 32)

    x[:, :, y1:y2, x1:x2] = x[rand_index, :, y1:y2, x1:x2]
    lam = 1 - (x2 - x1) * (y2 - y1) / (32 * 32)

    return x, y_a, y_b, lam

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)

        # Apply CutMix ~50% of the time
        if np.random.rand() > 0.5:
            inputs, targets_a, targets_b, lam = cutmix_data(inputs, targets)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = lam * criterion(outputs, targets_a) + (1 - lam) * criterion(outputs, targets_b)
        else:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()
        total_loss += loss.item() * inputs.size(0)
        correct    += outputs.argmax(1).eq(targets).sum().item()
        total      += inputs.size(0)
    return total_loss / total, 100. * correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        total_loss += loss.item() * inputs.size(0)
        correct    += outputs.argmax(1).eq(targets).sum().item()
        total      += inputs.size(0)
    return total_loss / total, 100. * correct / total

In [21]:
wandb.init(
    project="cnn-compression",
    name="structured-50pct-finetune",
    config={
        "epochs": EPOCHS,
        "lr": 1e-3,
        "weight_decay": 5e-4,
        "label_smoothing": 0.1,
        "pruning_ratio": 0.5,
        "optimizer": "SGD",
        "scheduler": "cosine",
    }
)

best_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, trainloader, optimizer, criterion, device)
    val_loss,   val_acc   = evaluate(model, testloader, criterion, device)
    scheduler.step()

    wandb.log({
        "epoch": epoch,
        "train/loss": train_loss,
        "train/acc": train_acc,
        "val/loss": val_loss,
        "val/acc": val_acc,
        "lr": scheduler.get_last_lr()[0],
    })

    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}% | "
          f"LR: {scheduler.get_last_lr()[0]:.6f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model, f'/content/practicum-project/models/structured_pruning/pruned/structured_pruned_50pct_fp32_finetuned{EPOCHS}.pth')
        wandb.summary["best_val_acc"] = best_acc
        print(f"  → Saved (best val acc: {best_acc:.2f}%)")

wandb.finish()

epoch,▁
lr,▁
train/acc,▁
train/loss,▁
val/acc,▁
val/loss,▁
epoch,1
lr,0.001
train/acc,80.17
train/loss,1.06957
val/acc,91.63


Epoch 01/40 | Train Loss: 1.0466 Acc: 80.82% | Val Loss: 0.7021 Acc: 92.04% | LR: 0.000994
  → Saved (best val acc: 92.04%)
Epoch 02/40 | Train Loss: 1.0242 Acc: 81.69% | Val Loss: 0.6985 Acc: 92.35% | LR: 0.000986
  → Saved (best val acc: 92.35%)
Epoch 03/40 | Train Loss: 1.0239 Acc: 79.99% | Val Loss: 0.6847 Acc: 92.62% | LR: 0.000976
  → Saved (best val acc: 92.62%)
Epoch 04/40 | Train Loss: 0.9828 Acc: 82.34% | Val Loss: 0.6898 Acc: 92.96% | LR: 0.000962
  → Saved (best val acc: 92.96%)
Epoch 05/40 | Train Loss: 0.9732 Acc: 81.24% | Val Loss: 0.6765 Acc: 93.10% | LR: 0.000946
  → Saved (best val acc: 93.10%)
Epoch 06/40 | Train Loss: 1.0118 Acc: 80.10% | Val Loss: 0.6907 Acc: 92.98% | LR: 0.000926
Epoch 07/40 | Train Loss: 0.9783 Acc: 83.57% | Val Loss: 0.6771 Acc: 93.25% | LR: 0.000905
  → Saved (best val acc: 93.25%)
Epoch 08/40 | Train Loss: 0.9566 Acc: 83.23% | Val Loss: 0.6677 Acc: 93.29% | LR: 0.000880
  → Saved (best val acc: 93.29%)
Epoch 09/40 | Train Loss: 0.9305 Acc: 83.

epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
lr,██████▇▇▇▇▇▆▆▆▆▅▅▅▅▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁
train/acc,▂▂▁▃▂▁▄▄▄▅▄▄▄▅▆▃▅▅▆▅▄▅▄▄▆▅▅▆▆▅▆▅▇█▅▅▄▅▅▅
train/loss,█▇▇▅▅▇▅▄▃▄▄▄▄▃▂▅▃▂▃▂▄▂▄▃▃▃▃▁▂▃▃▂▂▁▂▃▃▃▂▂
val/acc,▁▂▃▄▄▄▅▅▅▅▆▆▆▆▇▆▇▇▇▇▇▇▇▇▇▇▇▇█▇███▇██▇█▇█
val/loss,██▆▆▅▇▅▄▃▃▃▃▃▅▂▃▂▂▂▂▄▂▅▃▃▃▅▂▂▄▂▂▁▂▁▂▅▃▄▁
best_val_acc,94.35
epoch,40
lr,0.0
train/acc,84.4
train/loss,0.90388


In [24]:
from torch.ao.quantization import get_default_qconfig
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx
import time
import tracemalloc

device = torch.device('cpu')

model_fp32 = torch.load(
    f'/content/practicum-project/models/structured_pruning/pruned/structured_pruned_50pct_fp32_finetuned{EPOCHS}.pth',
    map_location='cpu',
    weights_only=False
).eval()

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])
calib_set = torchvision.datasets.CIFAR10(root='./data', train=True,
                                          download=False, transform=transform_test)
calib_loader = torch.utils.data.DataLoader(calib_set, batch_size=64,
                                            shuffle=False, num_workers=2)

qconfig = get_default_qconfig('fbgemm')
qconfig_dict = {"": qconfig}

example_input = torch.randn(1, 3, 32, 32)
model_prepared = prepare_fx(model_fp32, qconfig_dict, example_input)

print("Calibrating...")
model_prepared.eval()
with torch.no_grad():
    for i, (inputs, _) in enumerate(calib_loader):
        model_prepared(inputs)
        if i >= 15:  # 16 batches * 64 = ~1024 images, enough
            break

model_int8 = convert_fx(model_prepared)
print("Quantization done.")

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                        download=False, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=256,
                                          shuffle=False, num_workers=2)

correct, total = 0, 0
model_int8.eval()
with torch.no_grad():
    for inputs, targets in testloader:
        outputs = model_int8(inputs)
        correct += outputs.argmax(1).eq(targets).sum().item()
        total   += inputs.size(0)
print(f"INT8 Accuracy: {100. * correct / total:.2f}%")

dummy = torch.randn(1, 3, 32, 32)
# warmup
for _ in range(20):
    model_int8(dummy)

times = []
for _ in range(100):
    start = time.perf_counter()
    model_int8(dummy)
    times.append((time.perf_counter() - start) * 1000)
print(f"INT8 Latency: {sum(times)/len(times):.2f} ms")

save_path = '/content/practicum-project/models/structured_pruning/pruned_and_quantized/structured_pruned_50pct_finetuned_int8.pt'
torch.save(model_int8, save_path)

import os
size_mb = os.path.getsize(save_path) / (1024 ** 2)
print(f"INT8 Size: {size_mb:.2f} MB")

Calibrating...


/tmp/ipykernel_1744/3540562908.py:28: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_prepared = prepare_fx(model_fp32, qconfig_dict, example_input)
/usr/local/lib/python3.12/dist-packages/torch/ao/quantization/quantize_fx.py:146: FutureWarning: Passing a QConfig dictionary to prepare is deprecated and will not be supported in a future version. Please pass in a QConf

Quantization done.
INT8 Accuracy: 94.32%
INT8 Latency: 10.39 ms
INT8 Size: 2.76 MB


In [27]:
model_fp32.eval()
dummy = torch.randn(1, 3, 32, 32)

# warmup
for _ in range(20):
    model_fp32(dummy)

times = []
for _ in range(100):
    start = time.perf_counter()
    model_fp32(dummy)
    times.append((time.perf_counter() - start) * 1000)

print(f"FP32 Latency: {sum(times)/len(times):.2f} ms")

FP32 Latency: 13.30 ms
